## Chirps Data

In [ ]:
# !python -m pip install tqdm --quiet

In [2]:
# conda install geopandas rasterio rasterstats -c conda-forge

import os
import re
import glob
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from tqdm import tqdm
from rasterstats import zonal_stats
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns
from datetime import datetime
from rasterio.mask import mask
from scipy.stats import zscore
import rasterio

## Estimating Chirps Data for Adm Areas


In [4]:
def process_year_folder(base_folder, year, geojson_path, output_filename=None):
    """
    Process all TIFF files in a specified year folder.
    
    Args:
        base_folder (str): Path to the parent folder containing year folders
        year (int/str): Year to process (e.g., 2021 or "2021")
        geojson_path (str): Path to adm_4 GeoJSON file
        output_filename (str, optional): Custom output filename. Defaults to "precipitation_{year}.csv"
    """
    # Convert year to string if needed
    year = str(year)
    
    # Set up paths
    year_folder = os.path.join(base_folder, year)
    if output_filename is None:
        output_filename = f"precipitation_{year}.csv"
    output_path = os.path.join(base_folder, output_filename)
    
    # Verify input paths
    if not os.path.exists(year_folder):
        print(f"Error: Year folder not found at {year_folder}")
        return
    if not os.path.exists(geojson_path):
        print(f"Error: GeoJSON file not found at {geojson_path}")
        return

    # Load administrative boundaries
    try:
        gdf = gpd.read_file(geojson_path)
        print(f"Loaded GeoJSON with {len(gdf)} administrative boundaries")
    except Exception as e:
        print(f"Failed to load GeoJSON: {str(e)}")
        return

    # Get list of TIFF files for the specified year
    tiff_files = [f for f in os.listdir(year_folder) 
                 if f.endswith('.tif') and f.startswith(f'chirps-v3.0.{year}.')]
    
    if not tiff_files:
        print(f"No TIFF files found for {year} in {year_folder}")
        return

    print(f"\nFound {len(tiff_files)} TIFF files to process for year {year}")

    all_results = []
    
    for tiff_file in tqdm(tiff_files, desc=f"Processing {year} files"):
        tiff_path = os.path.join(year_folder, tiff_file)
        
        try:
            # Extract date from filename
            parts = tiff_file.split('.')
            file_year = parts[2]
            month = int(parts[3])
            date_str = f"{file_year}-{month:02d}"
            season = "wet" if 6 <= month <= 10 else "dry"

            with rasterio.open(tiff_path) as src:
                # Reproject polygons if needed
                if gdf.crs != src.crs:
                    gdf_reproj = gdf.to_crs(src.crs)
                else:
                    gdf_reproj = gdf

                for idx, row in gdf_reproj.iterrows():
                    geom = [row['geometry']]
                    
                    try:
                        # Mask the raster with the polygon
                        out_image, out_transform = mask(src, geom, crop=True, nodata=src.nodata)
                        
                        # Get precipitation values
                        values = out_image.flatten()
                        if src.nodata is not None:
                            values = values[values != src.nodata]
                        
                        # Calculate statistics
                        stats = {
                            'precip_median': float(np.median(values)) if len(values) > 0 else None,
                            'precip_sum': float(np.sum(values)) if len(values) > 0 else None,
                            'precip_max': float(np.max(values)) if len(values) > 0 else None,
                            'pixel_count': len(values) if len(values) > 0 else 0,
                            'month': month,
                            'season': season,
                            'tiff_file': tiff_file
                        }
                        
                        # Create result record (preserve all polygon attributes)
                        result = {**row.drop('geometry').to_dict()}
                        result.update(stats)
                        all_results.append(result)
                        
                    except Exception as e:
                        print(f"\nError processing polygon {idx} in {tiff_file}: {str(e)}")
                        continue
        
        except Exception as e:
            print(f"\nError processing {tiff_file}: {str(e)}")
            continue

    # Create and save DataFrame
    if all_results:
        df = pd.DataFrame(all_results)
        
        # Add year column (in case of multi-year processing)
        df['year'] = year
        
        # Save to CSV
        df.to_csv(output_path, index=False)
        print(f"\nSuccessfully processed {len(df)} records")
        print(f"Results saved to: {output_path}")
        
        # Show summary statistics
        print("\nMonthly precipitation summary:")
        summary = df.groupby(['month', 'season']).agg({
            'precip_median': 'median',
            'precip_sum': 'sum',
            'pixel_count': 'sum'
        })
        print(summary)
        
        return df
    else:
        print("\nNo data was extracted - empty results")
        return pd.DataFrame()



In [5]:

for year in range(2021, 2025):
      # Adjust range as needed
    if __name__ == "__main__":
        # Configuration - change these values
        BASE_FOLDER = r"D:\CDR_study_areas\precipitation"
        YEAR_TO_PROCESS = year  # Change this to the year you want to process
        GEOJSON_PATH = r"D:\CDR_study_areas\study_areas\seng_adm_4\communes_SNG.geojson"
        
        # Process the specified yea
        results_df = process_year_folder(
            base_folder=BASE_FOLDER,
            year=YEAR_TO_PROCESS,
            geojson_path=GEOJSON_PATH
        )


Loaded GeoJSON with 553 administrative boundaries

Found 12 TIFF files to process for year 2021


Processing 2021 files: 100%|██████████| 12/12 [00:10<00:00,  1.19it/s]



Successfully processed 6636 records
Results saved to: D:\CDR_study_areas\precipitation\precipitation_2021.csv

Monthly precipitation summary:
              precip_median    precip_sum  pixel_count
month season                                          
1     dry               0.0  1.914974e+03        17979
2     dry               0.0  2.154880e+03        17979
3     dry               0.0  5.262467e+01        17979
4     dry               0.0  2.889587e+02        17979
5     dry               0.0  1.433532e+04        17979
6     wet               0.0  4.444575e+05        17979
7     wet               0.0  7.988184e+05        17979
8     wet               0.0  2.191784e+06        17979
9     wet               0.0  1.094581e+06        17979
10    wet               0.0  4.122367e+05        17979
11    dry               0.0  5.756405e+03        17979
12    dry               0.0  5.941192e+02        17979
Loaded GeoJSON with 553 administrative boundaries

Found 12 TIFF files to process for y

Processing 2022 files: 100%|██████████| 12/12 [00:10<00:00,  1.18it/s]



Successfully processed 6636 records
Results saved to: D:\CDR_study_areas\precipitation\precipitation_2022.csv

Monthly precipitation summary:
              precip_median    precip_sum  pixel_count
month season                                          
1     dry               0.0  1.070935e+03        17979
2     dry               0.0  3.116340e+02        17979
3     dry               0.0  1.815493e+02        17979
4     dry               0.0  2.539047e+02        17979
5     dry               0.0  1.910043e+05        17979
6     wet               0.0  6.727650e+05        17979
7     wet               0.0  1.043433e+06        17979
8     wet               0.0  1.405433e+06        17979
9     wet               0.0  1.375919e+06        17979
10    wet               0.0  5.384156e+05        17979
11    dry               0.0  9.855976e+03        17979
12    dry               0.0  8.576791e+02        17979
Loaded GeoJSON with 553 administrative boundaries

Found 12 TIFF files to process for y

Processing 2023 files: 100%|██████████| 12/12 [00:09<00:00,  1.21it/s]



Successfully processed 6636 records
Results saved to: D:\CDR_study_areas\precipitation\precipitation_2023.csv

Monthly precipitation summary:
              precip_median    precip_sum  pixel_count
month season                                          
1     dry               0.0  1.591194e+03        17979
2     dry               0.0  3.903046e+02        17979
3     dry               0.0  1.016774e+02        17979
4     dry               0.0  1.590454e+03        17979
5     dry               0.0  1.058257e+05        17979
6     wet               0.0  5.868203e+05        17979
7     wet               0.0  1.262301e+06        17979
8     wet               0.0  1.441001e+06        17979
9     wet               0.0  1.204905e+06        17979
10    wet               0.0  3.922958e+05        17979
11    dry               0.0  4.584338e+02        17979
12    dry               0.0  1.661139e+03        17979
Loaded GeoJSON with 553 administrative boundaries

Found 12 TIFF files to process for y

Processing 2024 files: 100%|██████████| 12/12 [00:10<00:00,  1.18it/s]


Successfully processed 6636 records
Results saved to: D:\CDR_study_areas\precipitation\precipitation_2024.csv

Monthly precipitation summary:
              precip_median    precip_sum  pixel_count
month season                                          
1     dry               0.0  2.159511e+02        17979
2     dry               0.0  7.743736e+02        17979
3     dry               0.0  1.259984e+02        17979
4     dry               0.0  1.677136e+02        17979
5     dry               0.0  8.522443e+04        17979
6     wet               0.0  2.476729e+05        17979
7     wet               0.0  1.266877e+06        17979
8     wet               0.0  1.222060e+06        17979
9     wet               0.0  1.548476e+06        17979
10    wet               0.0  7.078707e+05        17979
11    dry               0.0  2.201619e+03        17979
12    dry               0.0  7.151816e+03        17979
